# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді:
1. Словник і merges практично не змінились би: ByteLevel pre-tokenizer розбиває текст на слова за регулярним виразом, тому злиття пар ніколи не перетинають межі pre-token'ів- і межу віршів теж. Ламається інше - ресурси: один суцільний рядок треба цілком тримати в пам'яті й попередньо токенізувати за один прохід, а train_from_iterator не може розпаралелити роботу між елементами, бо елемент лише один.
2. Токенізація складається з чотирьох етапів:
   1. нормалізація - очищення тексту: Unicode-нормалізація, нижній регістр, прибирання акцентів і зайвих пробілів;
   2. попередня токенізація - розбиття тексту на «слова»/pre-token'и за пробілами, пунктуацією чи регулярним виразом; токени надалі не перетинають межі pre-token'ів;
   3. модель токенізації - розбиття кожного pre-token'а на підслова вивченим алгоритмом (BPE / WordPiece / Unigram) і перетворення підслів на id зі словника;
   4. постобробка - додавання спеціальних токенів ([CLS], [SEP], \<eos> тощо), формування attention mask / token type ids, обрізання й паддінг.
3. Токен \<unk> позначає невідомі токени, відсутні у словнику, а \<eos> позначає кінець окремого вірша чи документа.
4. Частка \<unk> не зростає, оскільки зберігається базовий алфавіт. Довжина послідовності збільшується (слова дробляться на дрібніші частини), тому модель бачить менше тексту за один CONTEXT_LEN.
5. Він розділяє окремі вірші; без нього кінцівка одного вірша злиється з початком наступного, що зламає авторегресійне навчання генерації.
6.  train_data і val_data — одновимірні тензори PyTorch типу torch.int64: послідовні зрізи (90% / 10%) єдиного закодованого корпусу, train_data.shape = [19 454 696], val_data.shape = [2 161 633] (за виводом демо). Кожен елемент - id токена зі словника BPE; вірші йдуть один за одним, розділені id токена \<eos>. Тип int64 потрібен, бо ці id подаються в nn.Embedding.
7. Було додано assert torch.equal(xb[:, 1:], yb[:, :-1])
8. Нижньотрикутна матриця дозволяє позиції t бачити тільки поточний та попередні токени. Без маски модель отримає можливість підглядати майбутні токени під час тренування. Втрати швидко впадуть до нуля, але модель не навчиться прогнозувати наступні токени і під час генерації видаватиме хаотичний текст або нескінченні повторення.
9. Збігаються: Коли модель стабільно покращує val_loss до останньої ітерації.
Не збігаються: Коли модель починає перенавчатися, і val_loss починає зростати або коливатися наприкінці тренування.За результатами тренування найкращий чекпоінт досягнуто на iter 7850 (val_loss=4.920), тоді як остання ітерація була 7999 (val_loss=4.931). Отже, ці два чекпоінти не збігаються через незначні коливання втрат наприкінці.
10.  Зменшення інтервалу оцінки робить криву навчання значно детальнішою на ранніх ітераціях, але збільшує загальний час тренування. Подальше зменшення недоцільне, оскільки обчислення estimate_loss() починає перевантажувати процес тренування та додавати зайвий шум на графік.
11. Температура T ділить логіти zi / T у формулі Softmax, масштабуючи контрастність розподілу ймовірностей. Низька температура (T=0.2) робить розподіл гострим для вибору лише найімовірніших слів (детермінований текст), тоді як висока (T=1.4) згладжує його, підвищуючи шанс вибору випадкових і хаотичних токенів.
12. Параметр top_p обмежує семплювання найменшою підмножиною найімовірніших токенів, сумарна ймовірність яких сягає порогу $p$. Це відфільтровує низькоймовірні варіанти зі словника та запобігає появі некоректних або невлучних слів у тексті.
13. При ввімкненні enable_thinking=True модель перед формулюванням остаточної відповіді додає блок внутрішніх міркувань, обгорнутий у теги \<think>...\</think>.
У моєму прогоні 4.2 для всіх трьох пар (temperature, top_p) відповідь складається лише з міркувань англійською, хоча промпт українською, і обривається на півслові, бо max_new_tokens=300 вичерпується ще всередині \<think>; самого продовження речення немає. Міркування містять і фактичну помилку: модель називає Житомирську політехніку то Харківською, то Хмельницькою. Отже, за такого бюджету thinking-режим не покращив результат, а зіпсував його: усі токени пішли на планування.

---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

Обранний датасет Ukrainian news

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import kagglehub

path = kagglehub.dataset_download("zepopo/ukrainian-fake-and-true-news")
print("Датасет завантажено в папку:", path)
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.read_csv(os.path.join(path, csv_files[0]))
texts = df['Text'].dropna().astype(str).tolist()
corpus_text = "\n\n".join(texts)
print(f"Всього новин у датасеті: {len(df):,}")
print(f"Загальний розмір корпусу: {len(corpus_text):,} символів")
print("Приклад першої новини:\n", texts[0][:300])

Using Colab cache for faster access to the 'ukrainian-fake-and-true-news' dataset.
Датасет завантажено в папку: /kaggle/input/ukrainian-fake-and-true-news
Всього новин у датасеті: 50,030
Загальний розмір корпусу: 14,292,702 символів
Приклад першої новини:
 В Україні запустять механізм перевірки наявності подвійного або множинного громадянства. Особливо це стосуватиметься чиновників та депутатів. https://bit.ly/2P4NL8v


---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

In [2]:
# 2.1
from transformers import AutoTokenizer

base_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
new_tokenizer = base_tokenizer.train_new_from_iterator(text_iterator=texts, vocab_size=500)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

SPECIAL_TOKENS = ["<unk>", "<eos>"]
VOCAB_SIZE = 500

bpe_tok = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tok.pre_tokenizer = pre_tokenizers.Whitespace()
bpe_tok.train_from_iterator(texts, trainer=trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS))

wp_tok = Tokenizer(models.WordPiece(unk_token="<unk>"))
wp_tok.pre_tokenizer = pre_tokenizers.Whitespace()
wp_tok.train_from_iterator(texts, trainer=trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS))

uni_tok = Tokenizer(models.Unigram())
uni_tok.pre_tokenizer = pre_tokenizers.Whitespace()
uni_tok.train_from_iterator(texts, trainer=trainers.UnigramTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, unk_token="<unk>"))


In [4]:
# 2 ( 1.1 )
sample_text = "У Лісабоні українки світу обговорили, як протидіяти дезінформації та зміцнювати довіру до України"
test_freq = "Україна"
test_rare = "дезінформація"
comparison = []
for name, tok in [("BPE", bpe_tok), ("WordPiece", wp_tok), ("Unigram", uni_tok)]:
    total_tokens = len(tok.encode(sample_text).ids)
    comparison.append({"Алгоритм": name, "Розмір словника": tok.get_vocab_size(), "Кількість токенів у зразку": total_tokens, f"Токени ('{test_freq}')": " | ".join(tok.encode(test_freq).tokens), f"Токени ('{test_rare}')": " | ".join(tok.encode(test_rare).tokens)})
df_comparison = pd.DataFrame(comparison)
print(df_comparison.to_markdown(index=False))

| Алгоритм   |   Розмір словника |   Кількість токенів у зразку | Токени ('Україна')               | Токени ('дезінформація')                                                  |
|:-----------|------------------:|-----------------------------:|:---------------------------------|:--------------------------------------------------------------------------|
| BPE        |               500 |                           43 | Украї | на                       | де | з | ін | ф | ор | ма | ці | я                                        |
| WordPiece  |               500 |                           74 | У | ##к | ##ра | ##ї | ##н | ##а | д | ##е | ##з | ##і | ##н | ##ф | ##о | ##р | ##м | ##а | ##ц | ##і | ##я |
| Unigram    |               500 |                           47 | У | країн | а                    | де | з | і | н | форм | ація                                              |


Найкоротшу послідовність дав BPE (43), Unigram майже не відстає (47), WordPiece помітно довший (74) - через подвоєний за рахунок ## алфавіт і інший критерій злиття.
Це повністю узгоджується з туторіалом за рахунок підходу «знизу вгору» BPE і WordPiece будують словник знизу вгору від алфавіту. BPE зливає найчастіші пари, тому за малий бюджет швидко отримує довгі токени для частих фрагментів. WordPiece обирає пару за оцінкою freq(пари)/(freq(a)·freq(b)), тобто віддає перевагу парам із рідкісними складовими, а його алфавіт містить кожну літеру двічі (на початку слова і з префіксом ##). Тому при 500 токенах майже весь словник займає алфавіт, а на злиття майже нічого не лишається. Unigram, навпаки, йде згори вниз: починає з великого набору підрядків і викидає ті, що найменше погіршують правдоподібність корпусу. Через це він зберігає морфемоподібні фрагменти (форм, ація), корисні для рідкісних слів, хоча загалом на зразку дав трохи довшу послідовність, ніж BPE.

---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

In [5]:
#3.1
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from torch.utils.tensorboard import SummaryWriter

os.makedirs("models", exist_ok=True)
os.makedirs("runs", exist_ok=True)
results = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Використовуємо пристрій:", device)
VOCAB_SIZE_GPT = 8000
SPECIAL_TOKENS = ["<unk>", "<eos>"]
gpt_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
gpt_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
gpt_tokenizer.decoder = decoders.ByteLevel()
bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE_GPT, special_tokens=SPECIAL_TOKENS)
gpt_tokenizer.train_from_iterator(texts, trainer=bpe_trainer)
stoi = gpt_tokenizer.get_vocab()
itos = [None] * len(stoi)
for tok, idx in stoi.items():
    itos[idx] = tok
UNK_ID = stoi["<unk>"]
EOS_ID = stoi["<eos>"]
def tokenize(text: str) -> list[str]:
    return gpt_tokenizer.encode(text).tokens
def encode(tokens: list[str]) -> list[int]:
    return [stoi.get(t, UNK_ID) for t in tokens]
def decode(ids: list[int]) -> str:
    return gpt_tokenizer.decode(ids)
ENCODE_BATCH_SIZE = 1000
token_id_chunks = []
for start in range(0, len(texts), ENCODE_BATCH_SIZE):
    batch = texts[start:start + ENCODE_BATCH_SIZE]
    encodings = gpt_tokenizer.encode_batch(batch)
    for enc in encodings:
        token_id_chunks.append(np.array(enc.ids + [EOS_ID], dtype=np.int32))
token_ids = np.concatenate(token_id_chunks)
del token_id_chunks
gc.collect()
data = torch.from_numpy(token_ids.astype(np.int64))
del token_ids
gc.collect()
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
val_data = data[split_idx:]
print(f"Розмір словника BPE: {len(itos)}")
print(f"Всього токенів у корпусі новин: {len(data):,}")
print(f"Train токенів: {len(train_data):,}, Val токенів: {len(val_data):,}")

Використовуємо пристрій: cuda
Розмір словника BPE: 8000
Всього токенів у корпусі новин: 3,761,953
Train токенів: 3,385,757, Val токенів: 376,196


In [6]:
#3.2
class AttentionHead(nn.Module):
    def __init__(self, embed_dim: int, head_dim: int, context_len: int, dropout: float):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        causal_mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer("causal_mask", causal_mask)
    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        return weights @ v
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim, head_dim, context_len, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))
class FeedForward(nn.Module):
    def __init__(self, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(embed_dim, 4 * embed_dim),nn.ReLU(),nn.Linear(4 * embed_dim, embed_dim),nn.Dropout(dropout),)
    def forward(self, x):
        return self.net(x)
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_len: int, dropout: float):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.feed_forward = FeedForward(embed_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x
class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, n_heads: int, n_layers: int, context_len: int, dropout: float):
        super().__init__()
        self.context_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads, context_len, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 0.8, top_k: int | None = 40):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx
def get_batch(split: str, context_len: int, batch_size: int):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - context_len - 1, (batch_size,))
    x = torch.stack([source[i:i + context_len] for i in ix])
    y = torch.stack([source[i + 1:i + context_len + 1] for i in ix])
    return x.to(device), y.to(device)
def train_model(config: dict, model_name: str):
    print(f"\n=== Тренування {model_name} ===")
    model = MiniGPT(
        vocab_size=len(itos),
        embed_dim=config["embed_dim"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        context_len=config["context_len"],
        dropout=config["dropout"]
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
    writer = SummaryWriter(log_dir=os.path.join("runs", model_name))
    ckpt_path = os.path.join("models", f"{model_name}.pt")
    @torch.no_grad()
    def estimate_loss():
        model.eval()
        out = {}
        for split in ("train", "val"):
            losses = torch.zeros(config["eval_iters"])
            for i in range(config["eval_iters"]):
                xb, yb = get_batch(split, config["context_len"], config["batch_size"])
                _, loss = model(xb, yb)
                losses[i] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out
    best_val_loss = float("inf")
    start_time = time.time()
    for it in range(config["max_iters"]):
        xb, yb = get_batch("train", config["context_len"], config["batch_size"])
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if it % config["eval_interval"] == 0 or it == config["max_iters"] - 1:
            losses = estimate_loss()
            writer.add_scalar("Loss/train", losses["train"], it)
            writer.add_scalar("Loss/val", losses["val"], it)
            print(f"Iter {it:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                torch.save(model.state_dict(), ckpt_path)
    writer.close()
    print(f"Завершено за {(time.time() - start_time) / 60:.1f} хв. Найкращий Val Loss: {best_val_loss:.4f}")
    return model, best_val_loss
cfg_v1 = {
    "context_len": 64, "embed_dim": 128, "n_heads": 4, "n_layers": 4,
    "dropout": 0.1, "batch_size": 64, "learning_rate": 3e-4,
    "max_iters": 2000, "eval_interval": 250, "eval_iters": 50
}
cfg_v2 = {
    "context_len": 128, "embed_dim": 256, "n_heads": 8, "n_layers": 6,
    "dropout": 0.1, "batch_size": 64, "learning_rate": 3e-4,
    "max_iters": 2000, "eval_interval": 250, "eval_iters": 50
}
model_v1, best_val_v1 = train_model(cfg_v1, "minigpt_v1")
model_v2, best_val_v2 = train_model(cfg_v2, "minigpt_v2")
results.append({"Model / Variant": "mini-GPT v1","Params (M)": round(sum(p.numel() for p in model_v1.parameters() if p.requires_grad) / 1e6, 2),"val_loss": round(float(best_val_v1), 4),"Accuracy": None,"F1": None,"Inf Time (s)": None})
results.append({"Model / Variant": "mini-GPT v2","Params (M)": round(sum(p.numel() for p in model_v2.parameters() if p.requires_grad) / 1e6, 2), "val_loss": round(float(best_val_v2), 4),"Accuracy": None,"F1": None,"Inf Time (s)": None})


=== Тренування minigpt_v1 ===
Iter    0 | Train Loss: 9.1248 | Val Loss: 9.1265
Iter  250 | Train Loss: 7.3238 | Val Loss: 7.3087
Iter  500 | Train Loss: 6.8216 | Val Loss: 6.8444
Iter  750 | Train Loss: 6.4110 | Val Loss: 6.4344
Iter 1000 | Train Loss: 6.0951 | Val Loss: 6.1121
Iter 1250 | Train Loss: 5.8540 | Val Loss: 5.8840
Iter 1500 | Train Loss: 5.6612 | Val Loss: 5.7039
Iter 1750 | Train Loss: 5.4962 | Val Loss: 5.5320
Iter 1999 | Train Loss: 5.3466 | Val Loss: 5.4228
Завершено за 1.2 хв. Найкращий Val Loss: 5.4228

=== Тренування minigpt_v2 ===
Iter    0 | Train Loss: 8.9798 | Val Loss: 8.9818
Iter  250 | Train Loss: 6.5897 | Val Loss: 6.5972
Iter  500 | Train Loss: 5.7664 | Val Loss: 5.8010
Iter  750 | Train Loss: 5.2990 | Val Loss: 5.3414
Iter 1000 | Train Loss: 5.0205 | Val Loss: 5.1078
Iter 1250 | Train Loss: 4.8306 | Val Loss: 4.9204
Iter 1500 | Train Loss: 4.6801 | Val Loss: 4.8037
Iter 1750 | Train Loss: 4.5698 | Val Loss: 4.7016
Iter 1999 | Train Loss: 4.4935 | Val Los

In [7]:
#3.3
PROMPT = "У Києві"
def generate_sample(model, cfg, ckpt_name, prompt):
    model.load_state_dict(torch.load(os.path.join("models", f"{ckpt_name}.pt"), map_location=device))
    prompt_ids = torch.tensor([encode(tokenize(prompt))], dtype=torch.long).to(device)
    out_ids = model.generate(prompt_ids, max_new_tokens=50, temperature=0.7, top_k=49)
    return decode(out_ids[0].tolist())
print("=== Результати генерації ===")
print("Промпт:", PROMPT)
print("\n--- MiniGPT v1 (Val Loss: {:.4f}) ---".format(best_val_v1))
print(generate_sample(model_v1, cfg_v1, "minigpt_v1", PROMPT))
print("\n--- MiniGPT v2 (Val Loss: {:.4f}) ---".format(best_val_v2))
print(generate_sample(model_v2, cfg_v2, "minigpt_v2", PROMPT))

=== Результати генерації ===
Промпт: У Києві

--- MiniGPT v1 (Val Loss: 5.4228) ---
У Києві.️НАВ Україні, повітряна тривога! Повітряна тривога!️ У Львові! Повітряна тривога!У Києві оголошена повітряна тривога!️Голова ранку! У Росії: Путін з них: Міністерство оборони РФ️РВ Україні

--- MiniGPT v2 (Val Loss: 4.6062) ---
У Києві виявили 300 тисяч євро, яких у якому видно, які можуть отримати "вільйдем", - йдеться в сюжеті  https://bit.ly/3bjMIF У Київській області з'явилася інформація про коронавірус. Про


In [8]:
import gc
import torch
del model_v1, model_v2, train_data, val_data, data, corpus_text, texts, df
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU memory allocated: 0.02 GB


###  3.3
Найкращий val_loss: Варіант MiniGPT v2 показав кращий показник val_loss (4.6062 проти 5.4228 у v1).

Якість тексту: MiniGPT v2 генерує значно більш узгоджені новинні фрази та дотримується синтаксису.

Найбільший ефект: Ключовий вплив дало збільшення ємності мережі (збільшення embed_dim з 128 до 256 та кількості шарів n_layers з 4 до 6).

---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

### 4.0. Запустіть приклади виконання різних NLP задач

In [9]:
!pip install "transformers[sentencepiece]==5.18.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 32.4 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.21.0
    Uninstalling transformers-5.21.0:
      Successfully uninstalled transformers-5.21.0


In [10]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598046541213989}]
[{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [11]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.844595193862915, 0.11197695881128311, 0.04342786595225334]}

In [12]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

... (15 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to make a small change, using the techniques of the art of making a small change. This course will teach you how to make a small change, using the techniques of the art of making a small change. In the course, we will teach you how to make a small change, using the techniques of the art of making a small change. In the course, we will teach you how to make a small change, using the techniques of the art of making a small change. In the course, we will teach you how to make a small change, using the techniques of the art of making a small change. In the course, we will teach you how to make a small change, using the techniques of the art of making a small change. In the course, we will teach you how to make a small change. In the course, we will teach you how to make a small change. In the course, we will teach you how to make a small change. In the course, we will teach you how to make a small change. In the course, we will tea

In [13]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619743525981903,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052695631980896,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [14]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [15]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/490 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

{'answer': 'AVERAGE > 36542',
 'coordinates': [(0, 1)],
 'cells': ['36542'],
 'aggregator': 'AVERAGE'}

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

In [16]:
from transformers import pipeline

# 1. Sentiment Analysis
classifier = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual")
print("1. Sentiment Analysis:")
print(classifier("Запуск нового україномовного сервісу пройшов надзвичайно успішно та продуктивно!"))


config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

1. Sentiment Analysis:
[{'label': 'positive', 'score': 0.9855154752731323}]


In [17]:
from transformers import pipeline

# 2. Zero-shot Classification
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")
print("\n2. Zero-shot Classification:")
print(zero_shot("Уряд ухвалив рішення про виділення додаткових коштів на розвиток відновлюваної енергетики.",candidate_labels=["економіка", "спорт", "культура", "технології"]))


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

... (2 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]


2. Zero-shot Classification:
{'sequence': 'Уряд ухвалив рішення про виділення додаткових коштів на розвиток відновлюваної енергетики.', 'labels': ['технології', 'економіка', 'культура', 'спорт'], 'scores': [0.6008525490760803, 0.2760322391986847, 0.08613938838243484, 0.03697580099105835]}


In [18]:
from transformers import pipeline

# 3. Text Generation
generator = pipeline("text-generation", model="microsoft/Phi-4-mini-instruct")
print("\n3. Text Generation:")
print(generator("Основними перевагами використання мовних моделей є", max_length=50, num_return_sequences=1)[0]['generated_text'])

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

... (4 outputs left out)

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



3. Text Generation:
Основними перевагами використання мовних моделей є:

1. **Експертна інтерпретація**: Моделі можуть надати експертні інтерпретації даних, які можуть бути важливими для розробки стратегій маркетингу, управління бізнесу та розуміння клієнтів.

2. **Інтуїтивне розуміння**: Це допомагає організаціям розуміти, як вони використовують мову, що може допомогти підвищити ефективність комунікації.

3. **Гнучкість**: Моделі можуть обробляти новий контент, що робить їх незамінними при управлінні постійно змінюваною інформацією.

4. **Складність**: Ці моделі можуть обробляти складні операції, такі як перевірка версії мови, що можуть бути важко зробити вручну.

5. **Економічність**: Використання мовних моделей може знизити витрати на персонал для аналізу даних.

6. **Масштабованість**: Моделі можуть обробляти величезні


In [19]:
from transformers import pipeline

# 4. Fill-mask
unmasker = pipeline("fill-mask", model="Geotrend/bert-base-uk-cased")
print("\n4. Fill-mask:")
print(unmasker("Київ - це [MASK] України.", top_k=2))

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  381MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/98.5k [00:00<?, ?B/s]


4. Fill-mask:
[{'score': 0.07412812858819962, 'token': 2124, 'token_str': 'центр', 'sequence': 'Київ - це центр України.'}, {'score': 0.045155856758356094, 'token': 3448, 'token_str': 'місто', 'sequence': 'Київ - це місто України.'}]


In [20]:
from transformers import pipeline

# 5. Named Entity Recognition
ner = pipeline("ner", model="Babelscape/wikineural-multilingual-ner", aggregation_strategy="simple")
print("\n5. NER:")
print(ner("Володимир Зеленський зустрівся з представниками ЄС у Києві."))

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  709MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


5. NER:
[{'entity_group': 'PER', 'score': np.float32(0.9992399), 'word': 'Володимир Зеленський', 'start': 0, 'end': 20}, {'entity_group': 'ORG', 'score': np.float32(0.9911164), 'word': 'ЄС', 'start': 48, 'end': 50}, {'entity_group': 'LOC', 'score': np.float32(0.99889994), 'word': 'Києві', 'start': 53, 'end': 58}]


In [21]:
from transformers import pipeline
qa = pipeline("text-generation", model="microsoft/Phi-4-mini-instruct", device_map="auto")
table_md = """| Університет | Місто | Рік | Студентів |
|---|---|---|---|
| Житомирська політехніка | Житомир | 1920 | 9000 |
| КНУ ім. Шевченка | Київ | 1834 | 26000 |
| Львівська політехніка | Львів | 1844 | 35000 |"""
messages = [{"role": "user", "content": f"Таблиця:\n{table_md}\n\nСкільки студентів навчається в Житомирській політехніці? Відповідай одним числом."}]
print(qa(messages, max_new_tokens=20, do_sample=False)[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


9000


---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

In [22]:
import sys
import gc
import torch
cuda_vars = ['hf_model', 'model', 'model_v1', 'model_v2', 'classifier',
             'generator', 'unmasker', 'ner', 'zero_shot', 'qa_pipeline', 'pipe']
for var_name in cuda_vars:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"Вільно VRAM: {(torch.cuda.mem_get_info()[0]) / 1e9:.2f} GB з {(torch.cuda.mem_get_info()[1]) / 1e9:.2f} GB")

Вільно VRAM: 10.81 GB з 15.64 GB


In [23]:
!pip install -q evaluate scikit-learn

In [24]:
#5.1 Sentiment Analysis
import time
import evaluate
import pandas as pd
from transformers import pipeline
eval_data = [
    {"text": "Швидка доставка, якісний товар, задоволений покупкою на всі сто!", "label": "POSITIVE"},
    {"text": "Інтерфейс програми дуже зручний та зрозумілий навіть для початківців.", "label": "POSITIVE"},
    {"text": "Дякую підтримці за оперативне вирішення мого питання протягом десяти хвилин.", "label": "POSITIVE"},
    {"text": "Неймовірно смачна їжа та привітний персонал, обов'язково завітаємо ще!", "label": "POSITIVE"},
    {"text": "Оновлення додатка додало корисні функції, все працює бездоганно.", "label": "POSITIVE"},
    {"text": "Курс перевершив усі мої очікування, матеріал подано дуже доступно.", "label": "POSITIVE"},
    {"text": "Ноутбук працює тихо, швидко і відмінно тримає заряд акумулятора.", "label": "POSITIVE"},
    {"text": "Чудовий готель з гарним краєвидом та ідеальною чистотою в номерах.", "label": "POSITIVE"},
    {"text": "Жахливий сервіс, кур'єр запізнився на дві години і навіть не вибачився.", "label": "NEGATIVE"},
    {"text": "Після останнього оновлення застосунок постійно зависає і вилітає.", "label": "NEGATIVE"},
    {"text": "Товар прийшов бракований, а техпідтримка ігнорує мої звернення.", "label": "NEGATIVE"},
    {"text": "Марно витрачені гроші, якість матеріалу просто жахлива.", "label": "NEGATIVE"},
    {"text": "Замовлення скасували без пояснення причин і досі не повернули кошти.", "label": "NEGATIVE"},
    {"text": "Дуже повільний сайт, сторінки вантажаться хвилинами і видають помилки.", "label": "NEGATIVE"},
    {"text": "Їжа була холодною та несмачною, повне розчарування від закладу.", "label": "NEGATIVE"},
    {"text": "Техніка зламалася вже на другий день використання, категорично не рекомендую!", "label": "NEGATIVE"},
]
texts = [item["text"] for item in eval_data]
references = [1 if item["label"] == "POSITIVE" else 0 for item in eval_data]
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [25]:
# 5.2
def evaluate_model(model_name, threshold=0.5):
    pipe = pipeline("sentiment-analysis", model=model_name)
    num_params = sum(p.numel() for p in pipe.model.parameters()) / 1e6
    start = time.time()
    raw_preds = pipe(texts)
    inf_time = time.time() - start
    predictions = []
    for p in raw_preds:
        label = str(p['label']).upper()
        score = p['score']
        if label in ['POSITIVE', 'POS', 'LABEL_2', '4 STARS', '5 STARS']:
            p_pos = score
        elif label in ['NEGATIVE', 'NEG', 'LABEL_0', '1 STAR', '2 STARS']:
            p_pos = 1.0 - score
        else:
            p_pos = score
        pred_label = 1 if p_pos >= threshold else 0
        predictions.append(pred_label)
    acc = accuracy_metric.compute(predictions=predictions, references=references)['accuracy']
    f1 = f1_metric.compute(predictions=predictions, references=references, pos_label=1)['f1']
    return {"Model / Variant": model_name,"Params (M)": round(num_params, 1),"val_loss": None,"Accuracy": round(acc, 3),"F1": round(f1, 3),"Inf Time (s)": round(inf_time, 3)}
res_m1 = evaluate_model("cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual")
res_m2 = evaluate_model("lxyuan/distilbert-base-multilingual-cased-sentiments-student")

df_comparison = pd.DataFrame([res_m1, res_m2])
print("=== 5.2. Порівняння базових моделей ===")
display(df_comparison)

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

=== 5.2. Порівняння базових моделей ===


,Model / Variant,Params (M),val_loss,Accuracy,F1,Inf Time (s)
0,cardiffnlp/twitter-xlm-roberta-base-sentiment-...,278.0,None,0.938,0.941,0.709
1,lxyuan/distilbert-base-multilingual-cased-sent...,135.3,None,0.812,0.800,0.309


In [26]:
# 5.3.
BEST_MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
res_exp_default = evaluate_model(BEST_MODEL_NAME, threshold=0.5)
res_exp_default["Model / Variant"] += " (Threshold 0.5 - Baseline)"
res_exp_low = evaluate_model(BEST_MODEL_NAME, threshold=0.3)
res_exp_low["Model / Variant"] += " (Threshold 0.3)"
res_exp_high = evaluate_model(BEST_MODEL_NAME, threshold=0.7)
res_exp_high["Model / Variant"] += " (Threshold 0.7)"
results.extend([res_m1, res_m2, res_exp_default, res_exp_low, res_exp_high])
df_results = pd.DataFrame(results)
df_results.to_csv("results.csv", index=False)
print("=== 5.3. Підсумкова таблиця експериментів ===")
display(df_results)
print("\nФайл results.csv успішно оновлено у корінній папці!")

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

=== 5.3. Підсумкова таблиця експериментів ===


,Model / Variant,Params (M),val_loss,Accuracy,F1,Inf Time (s)
0,mini-GPT v1,2.86,5.4228,NaN,NaN,NaN
1,mini-GPT v2,8.87,4.6062,NaN,NaN,NaN
2,cardiffnlp/twitter-xlm-roberta-base-sentiment-...,278.00,NaN,0.938,0.941,0.709
3,lxyuan/distilbert-base-multilingual-cased-sent...,135.30,NaN,0.812,0.800,0.309
4,cardiffnlp/twitter-xlm-roberta-base-sentiment-...,278.00,NaN,0.938,0.941,0.849
5,cardiffnlp/twitter-xlm-roberta-base-sentiment-...,278.00,NaN,0.938,0.941,0.511
6,cardiffnlp/twitter-xlm-roberta-base-sentiment-...,278.00,NaN,0.938,0.941,0.988



Файл results.csv успішно оновлено у корінній папці!


Найкраща модель: cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual (278M параметрів).

Вплив налаштувань: Вибір базової архітектури виявився вирішальним чинником. Налаштування порогу threshold дозволяє гнучко регулювати чутливість класифікатора, проте принциповий рівень точності визначається саме потужністю та обсягом навчальних даних моделі.

---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

In [27]:
!pip install -q gradio transformers torch

In [28]:
import gradio as gr
import torch
from transformers import pipeline

sentiment_analyzer = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual")
text_generator = pipeline( "text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=120, do_sample=True, temperature=0.6, top_p=0.9)
def process_feedback(feedback_text: str, sender_name: str, company_name: str):
    if not feedback_text.strip():
        return "Будь ласка, введіть текст відгуку.", ""
    raw_sentiment = sentiment_analyzer(feedback_text)[0]
    label = str(raw_sentiment['label']).lower()
    score = raw_sentiment['score']
    if 'pos' in label or 'positive' in label or 'label_2' in label:
        sentiment_str = f"Позитивний (впевненість: {score:.1%})"
        tone_instruction = "подякуй за позитивний відгук, вислови радість та побажай гарного дня"
    elif 'neg' in label or 'negative' in label or 'label_0' in label:
        sentiment_str = f"Негативний (впевненість: {score:.1%})"
        tone_instruction = "вибачся за незручності, запевни у вирішенні проблеми та запропонуй бонусну компенсацію"
    else:
        sentiment_str = f"Нейтральний (впевненість: {score:.1%})"
        tone_instruction = "подякуй за зворотний зв'язок та запевни у постійному покращенні сервісу"
    prompt = (
        f"<|im_start|>system\n"
        f"Ти - ввічливий менеджер з підтримки клієнтів компанії '{company_name}'. "
        f"Напиши коротку, професійну відповідь українською мовою на відгук клієнта {sender_name}. "
        f"Інструкція: {tone_instruction}.\n<|im_end|>\n"
        f"<|im_start|>user\nВідгук від {sender_name}: \"{feedback_text}\"\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    gen_out = text_generator(prompt)
    full_response = gen_out[0]['generated_text']
    response_text = full_response.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    return sentiment_str, response_text
demo = gr.Interface(
    fn=process_feedback,
    inputs=[
        gr.Textbox(
            lines=4,
            placeholder="Введіть текст відгуку...",
            label="Текст відгуку клієнта",
            value="Кур'єр запізнився на годину, а замовлення було холодним. Дуже розчарований сервісом!"
        ),
        gr.Textbox(label="Ім'я клієнта", value="Владислав"),
        gr.Textbox(label="Назва вашої компанії", value="ZTU")
    ],
    outputs=[gr.Textbox(label="Аналіз тональності"),gr.Textbox(label="Автоматично згенерована відповідь підтримки", lines=5)],
    title="SmartSupport: Автоматизація відповідей на відгуки",
    description="Застосунок визначає емоційний тон відгуку клієнта та автоматично формує готові відповіді",
    theme="soft"
)
demo.launch(share=True)

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1739: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1771: FutureWarning: split_attention_implementation is deprecated as the 'paged|' prefix is no longer needed for flash or SDPA.
  is_paged, base_implementation = split_attention_implementation(attn_implementation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8258d63bc5604f8c9b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```